# 04 - Pitcher fit scores and free-agent targets

Each pitcher's mix of batted balls (by batter side and type) is weighted by each park's effects.

- `edge_pts`: expected wOBA points on contact vs. a league-average park (positive helps the pitcher)
- `fit_pts`: `edge_pts` with the park's overall effect and the pitcher's overall tendency removed, so it says how much better this park suits him than it suits a typical pitcher (higher is a better fit); use this to rank targets. Without removing the park's overall effect, every pitcher would just inherit the park factor.

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
from src.config import PROCESSED, INTERIM
from src.park_effects import contact_frame
from src.fit import pitcher_fit, best_pitchers_for_park

df = pd.read_parquet(INTERIM / "pitches_archetyped.parquet")
contact = contact_frame(df)
effects = pd.read_csv(PROCESSED / "park_bb_effects.csv")

names = contact.drop_duplicates("pitcher").set_index("pitcher")["player_name"]
fit = pitcher_fit(contact, effects, min_bip=300)
print(f"{fit['pitcher'].nunique()} pitchers with 300+ balls in play")
fit.head()

## Best-fit pitchers for a park

In [ ]:
best_pitchers_for_park(fit, "Coors Field", names)

In [ ]:
best_pitchers_for_park(fit, "Oracle Park", names)

## Free-agent class

`data/processed/free_agents_2026.csv` lists this winter's free-agent pitchers (columns `name`, `role`, `team_2026`, `notes`). Names use Statcast's "Last, First" format, and the notebook looks up each pitcher's ID from the data, ignoring accents. Edit the CSV as signings happen.

Pitchers without enough MLB data (for example, players coming from Japan or Korea) will not match and are listed below.

In [ ]:
import unicodedata

def key(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()
    return " ".join(s.replace(".", "").replace(",", " ").split())

lookup = {key(n): p for p, n in names.items()}
fa = pd.read_csv(PROCESSED / "free_agents_2026.csv")
fa["mlbam_id"] = fa["name"].map(lambda n: lookup.get(key(n)))
print(f"Matched {fa['mlbam_id'].notna().sum()} of {len(fa)} free agents to Statcast IDs")
print("Not matched:", fa.loc[fa["mlbam_id"].isna(), "name"].tolist())

fa = fa.dropna(subset=["mlbam_id"]).astype({"mlbam_id": int})
fa_fit = fit.merge(fa, left_on="pitcher", right_on="mlbam_id")
print("No fit score (under 300 balls in play):", sorted(set(fa["name"]) - set(fa_fit["name"])))

### Best parks for each free agent

In [ ]:
best_parks = (fa_fit.sort_values("fit_pts", ascending=False).groupby("name").head(3)
                    .sort_values(["name", "fit_pts"], ascending=[True, False]))
display(best_parks[["name", "role", "park", "fit_pts"]].round(1))

### Best free agents for each park

In [ ]:
by_park = (fa_fit.sort_values("fit_pts", ascending=False).groupby("park").head(3)
                 .sort_values(["park", "fit_pts"], ascending=[True, False]))
by_park[["park", "name", "role", "fit_pts"]].round(1).to_csv(PROCESSED / "fa_targets_by_park.csv", index=False)
display(by_park[["park", "name", "role", "fit_pts"]].round(1))

### Best and worst park for each free agent

In [ ]:
bw = (fa_fit.sort_values("fit_pts")
            .groupby("name")
            .agg(role=("role", "first"), worst_park=("park", "first"), worst=("fit_pts", "first"),
                 best_park=("park", "last"), best=("fit_pts", "last"))
            .round(1)
            .sort_values("best", ascending=False))
print(bw.to_string())

## Backtest plan

The claim to test: knowing a pitcher's batted-ball mix and the park's effects predicts how his results change when he changes parks better than a plain park factor does.

1. List pitchers who changed home parks (free agency or trade) between consecutive seasons in your window.
2. For each, recompute the park effects **without that pitcher's own balls in play** so he cannot predict himself. Use his batted-ball mix from the seasons before the move.
3. Prediction: his `edge_pts` at the new park minus at the old park. Baseline: the same difference using one park-wide number (average gap by park) instead of batted-ball-level effects.
4. Outcome: the change in his own `woba_gap` on contact from before to after the move.
5. Compare correlation and error for the model vs. the baseline. Report the result honestly, including if the edge is small. Moves are rare, so samples will be small.

In [ ]:
# TODO: backtest (see plan above)

## Commit

```
git add notebooks/04_fa_targets.ipynb data/processed
git commit -m "pitcher fit scores and free-agent targets"
git push
```